# D.R.O.N.A. - 10 End-to-End Evaluation

Full-system metrics across all four research contributions, assembled into one
report:

- **C1** retrieval quality (hybrid vs dense)
- **C2** bias detection P/R/F1 **and** bias-MITIGATION metrics
- **C3** gesture smoothness (keyframe baseline; ACT comparison if a checkpoint exists)
- **C4** Nepal-first citation ratio, latency, citation grounding

Each block degrades gracefully if its dependency (ChromaDB / Ollama / checkpoint)
is missing, so the notebook always runs.

In [1]:
import sys; import os, pathlib
# Run from the repo root so relative data paths (settings.*) resolve correctly.
_root = pathlib.Path.cwd()
while not (_root / 'drona' / '__init__.py').exists() and _root != _root.parent:
    _root = _root.parent
os.chdir(_root)
sys.path.insert(0, str(_root))
import warnings; warnings.filterwarnings('ignore')
import json, pandas as pd
from drona.utils.logging import setup_logging; setup_logging('WARNING')
from drona.evaluation.harness import EvaluationHarness

report = EvaluationHarness().run_all(c4_with_llm=False)
print('Contributions evaluated:')
for c in ['c1', 'c2', 'c3', 'c4']:
    print(f'  {c.upper()}:', 'present' if getattr(report, c) else 'skipped')

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Contributions evaluated:
  C1: present
  C2: present
  C3: present
  C4: present


In [2]:
# C3 gesture smoothness summary + ACT-vs-keyframe comparison if available.
if report.c3:
    print(f'Mean jerk (keyframe baseline): {report.c3.mean_jerk:.4f} rad/s^3')
    print(f'Mean path length: {report.c3.mean_path_length:.3f} rad')
    print('All within spec:', report.c3.all_within_spec)

try:
    from drona.interaction.sim_eval import evaluate_keyframe_baseline
    base = evaluate_keyframe_baseline()
    print(f'\nKeyframe success rate: {base.success_rate:.2f}, mean jerk: {base.mean_jerk:.4f}')
except Exception as exc:
    print('sim_eval note:', exc)

Mean jerk (keyframe baseline): 11.5851 rad/s^3
Mean path length: 0.998 rad
All within spec: True

Keyframe success rate: 1.00, mean jerk: 0.0005


In [3]:
# Bias-MITIGATION + citation grounding need live responses (Ollama). Demonstrate
# the harness wiring; populate with real responses when Ollama is available.
from drona.evaluation.bias_mitigation import evaluate_bias_mitigation
from drona.evaluation.citation_eval import evaluate_citations
from drona.evaluation.queries import C4_QUERIES

responses, cases = [], []
try:
    from drona.advising.engine import AdvisingEngine, make_query
    eng = AdvisingEngine()
    if eng._llm.is_available():
        for q in C4_QUERIES:
            resp = eng.advise(make_query(q.query_text))
            responses.append(resp)
            retrieved = [c for p in resp.pathways for c in p.citations]
            cases.append((resp, retrieved))
except Exception as exc:
    print('Live advising skipped:', exc)

if responses:
    bm = evaluate_bias_mitigation(responses)
    ce = evaluate_citations(cases)
    print('Bias mitigation:', json.dumps(bm.to_dict(), indent=2)[:600])
    print('Citation grounding:', json.dumps(ce.to_dict(), indent=2)[:400])
else:
    print('Start Ollama to compute live bias-mitigation + citation metrics.')

2026-06-27 22:14:21 | WARNING  | drona.advising.llm_client:is_available:143 - Ollama availability check failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download


Start Ollama to compute live bias-mitigation + citation metrics.


In [4]:
# Persist the report for the thesis appendix.
from pathlib import Path
out = Path('..') / 'data' / 'evaluation' / 'end_to_end_report.json'
report.save(out)
print('Saved:', out)

Saved: ..\data\evaluation\end_to_end_report.json


**Output:** a single JSON report covering C1–C4 plus the custom bias-mitigation
and citation-grounding harnesses. This is the artefact cited in the evaluation
chapter; re-run with ChromaDB populated and Ollama running for full numbers.